# In-Class Activity 9

### 1) from pyspark.sql import SparkSession
- Purpose: Import Spark’s main entry point for DataFrame/SQL work.
- Output/Side-effects: None (import only).

In [0]:
from pyspark.sql import SparkSession

### 2) spark = SparkSession.builder.appName('data_processing_tasks').getOrCreate()
- Purpose: Start (or reuse) a Spark session labeled data_processing_tasks.
- Key notes: getOrCreate() avoids duplicates; you’ll see Spark startup logs in output.
- Output/Side-effects: Creates the spark handle used everywhere else.

In [0]:
spark=SparkSession.builder.appName('data_processing_tasks'). getOrCreate()

### 3) import pyspark.sql.functions as F
- Purpose: Shorthand import for column/SQL helpers (e.g., F.col, F.when, F.lit).
- Output/Side-effects: None; enables concise transforms.
### 4) from pyspark.sql.types import *
- Purpose: Bring schema/data types into scope (e.g., StructType, StringType, IntegerType).
- When to use: Defining explicit schemas instead of type inference.
- Output/Side-effects: None.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *


### 5) Define a schema (example)
> schema = StructType() \
>   .add("Customer_main_type","string") \
>   .add("Region Code","string")
- Purpose: Declare column names + types for the DataFrame.
- Why it matters: Prevents wrong type inference and documents your data contract.
- Output/Side-effects: A StructType object stored in schema. 

### 6) Create a small DataFrame
> region_data = spark.createDataFrame(
>   [("Family with grownups","PN"),
>    ("Driven Growers","GJ"),
>    ("Conservative families","DD"),
>    ("Cruising Seniors","DL"),
>    ("Average Family ","MN"),
>    ("Living well","KA"),
>    ("Successful hedonists","JH"),
>    ("Retired and Religious","AX"),
>    ("Career Loners","HY"),
>    ("Farmers","JH")],
>   schema=schema
> )
- Purpose: Build a typed DataFrame from in-memory Python rows.
- Key notes: Row order is not guaranteed in distributed ops; schema enforces types.
- Output/Side-effects: New DataFrame region_data with 10 rows × 2 columns.


In [0]:
region_data = spark.createDataFrame([('Family with grownups','PN'),('Driven Growers','GJ'),('Conservative families','DD'),('Cruising Seniors','DL'),
('Average Family ','MN'),('Living well','KA'),('Successful hedonists','JH'),('Retired and Religious','AX'),('Career Loners','HY'),('Farmers','JH')],
schema=StructType().add("Customer_main_type","string").add("Region Code","string")) 

###7) Inspect results
> region_data.printSchema()
- Purpose: Verify column names/types/nullability.
- Output: Tree printout of the schema.
### 8) Preview rows:

> region_data.show()        _Tabular preview in text_

> region_data.display()   _Databricks rich table UI_

- Purpose: Quick content check (first 10 rows by default).
- Output: Table preview; .display() is notebook-friendly. 

In [0]:
region_data.display()

Customer_main_type,Region Code
Family with grownups,PN
Driven Growers,GJ
Conservative families,DD
Cruising Seniors,DL
Average Family,MN
Living well,KA
Successful hedonists,JH
Retired and Religious,AX
Career Loners,HY
Farmers,JH


In [0]:
customer=spark.createDataFrame([('Family with grownups',72000),('Driven Growers',19000),('Conservative families',59500),
('Cruising Seniors',29500),('Average Family ',69438),('Living well',92000),('Successful hedonists',28500),('Retired and Religious',25750),
('Career Loners',32900),('Farmers',73000)],schema=StructType().add("Customer_main_type","string").add("Avg_Salary","integer"))


In [0]:
customer.display()

Customer_main_type,Avg_Salary
Family with grownups,72000
Driven Growers,19000
Conservative families,59500
Cruising Seniors,29500
Average Family,69438
Living well,92000
Successful hedonists,28500
Retired and Religious,25750
Career Loners,32900
Farmers,73000


Databricks visualization. Run in Databricks to view.

### Join: `new_df = customer.join(region_data, on='Customer_main_type')`

**Purpose**  
Enrich the `customer` table with region info by matching on `Customer_main_type`.

**Inputs**  
- `customer` — base DataFrame (must include `Customer_main_type`)  
- `region_data` — lookup DataFrame mapping `Customer_main_type` → region fields (e.g., `Region Code`)

**Operation**  
- Performs an **inner join** on the shared column `Customer_main_type`.  
- Keeps only rows where the key exists in **both** DataFrames.

**Output**  
- **`new_df`**: all `customer` columns + matching columns from `region_data`.

**Common Variations**  
```python
# Keep all customers, add region info when available
customer.join(region_data, on='Customer_main_type', how='left')

# Keep all region rows (rare for lookups)
customer.join(region_data, on='Customer_main_type', how='right')

# Keep everything from both sides
customer.join(region_data, on='Customer_main_type', how='outer')


In [0]:
new_df=customer.join(region_data,on='Customer_main_type')

In [0]:
new_df.display()

Customer_main_type,Avg_Salary,Region Code
Family with grownups,72000,PN
Driven Growers,19000,GJ
Conservative families,59500,DD
Cruising Seniors,29500,DL
Average Family,69438,MN
Living well,92000,KA
Successful hedonists,28500,JH
Retired and Religious,25750,AX
Career Loners,32900,HY
Farmers,73000,JH


Databricks visualization. Run in Databricks to view.

### Group count by region
**Purpose**  
Count how many rows fall into each `Region Code`.

**Operation**  
- Group rows by `Region Code`.  
- Compute row counts for each group.  
- Display the result.

**Output**  
A two-column table: `Region Code | count` (in your run, each code showed `count = 1`).

**Notes**  
- Add `.orderBy("count", ascending=False)` to see the largest groups first.  
- Columns with spaces are fine in the API, but when referenced as strings in expressions use backticks: ``F.col("`Region Code`")``.


In [0]:

new_df.groupby("Region Code").count().show()


+-----------+-----+
|Region Code|count|
+-----------+-----+
|         PN|    1|
|         GJ|    1|
|         DD|    1|
|         DL|    1|
|         MN|    1|
|         KA|    1|
|         JH|    2|
|         AX|    1|
|         HY|    1|
+-----------+-----+



### Sum salary by customer type
**Purpose**  
Compute total `Avg_Salary` for each `customer_main_type` and handle null aggregates.

**Operation**  
- Group by `customer_main_type`.  
- Sum the `Avg_Salary` within each group.  
- Replace any null totals with `0`.  
- Show the result.

**Output**  
Two columns: `customer_main_type | sum(Avg_Salary)`.

**Notes**  
- `sum()` skips null values at the row level; `fillna(0)` applies to the aggregated result.  
- For clarity, alias the sum: `...agg(F.sum('Avg_Salary').alias('total_salary'))`.


In [0]:
customer.groupBy('Customer_main_type').sum('Avg_Salary').fillna(0).show()

+--------------------+---------------+
|  Customer_main_type|sum(Avg_Salary)|
+--------------------+---------------+
|Family with grownups|          72000|
|      Driven Growers|          19000|
|Conservative fami...|          59500|
|    Cruising Seniors|          29500|
|     Average Family |          69438|
|         Living well|          92000|
|Successful hedonists|          28500|
|Retired and Relig...|          25750|
|       Career Loners|          32900|
|             Farmers|          73000|
+--------------------+---------------+



### Sort aggregated salaries
**Purpose**  
Order segment totals to compare lowest → highest (or vice versa).

**Operation**  
- Group by `customer_main_type`.  
- Sum `Avg_Salary`.  
- Replace null totals with `0`.  
- Sort by the aggregated column.

**Output**  
Aggregated rows sorted by `sum(Avg_Salary)` (ascending by default).

**Notes**  
- Use an alias + explicit sort for readability:  
  - `alias('total_salary')` then `orderBy(F.col('total_salary').desc())` for highest first.  
- Add `.limit(10)` to show top-N segments.


In [0]:
customer.groupBy('Customer_main_type').sum('Avg_Salary').fillna(0).orderBy('sum(Avg_Salary)').show()

+--------------------+---------------+
|  Customer_main_type|sum(Avg_Salary)|
+--------------------+---------------+
|      Driven Growers|          19000|
|Retired and Relig...|          25750|
|Successful hedonists|          28500|
|    Cruising Seniors|          29500|
|       Career Loners|          32900|
|Conservative fami...|          59500|
|     Average Family |          69438|
|Family with grownups|          72000|
|             Farmers|          73000|
|         Living well|          92000|
+--------------------+---------------+



### Sort aggregated salaries (descending)
**Purpose**  
Show which `customer_main_type` has the highest total `Avg_Salary`, from highest → lowest.

**Operation**  
- Group by `customer_main_type`.  
- Sum `Avg_Salary` for each group.  
- Replace any null sums with `0`.  
- Order the results by the aggregated column **descending**.  
- Display the table.

**Output**  
A two-column table: `customer_main_type | sum(Avg_Salary)` sorted with the largest totals on top.

**Notes**  
- The sort key is the generated name `'sum(Avg_Salary)'`. For clarity, alias then sort:
  ```python
  from pyspark.sql import functions as F
  (customer.groupBy('customer_main_type')
           .agg(F.sum('Avg_Salary').alias('total_salary'))
           .fillna(0)
           .orderBy(F.col('total_salary').desc())
           .show())


In [0]:
customer.groupBy('Customer_main_type').sum('Avg_Salary').fillna(0).orderBy('sum(Avg_Salary)',ascending=False).show()

+--------------------+---------------+
|  Customer_main_type|sum(Avg_Salary)|
+--------------------+---------------+
|         Living well|          92000|
|             Farmers|          73000|
|Family with grownups|          72000|
|     Average Family |          69438|
|Conservative fami...|          59500|
|       Career Loners|          32900|
|    Cruising Seniors|          29500|
|Successful hedonists|          28500|
|Retired and Relig...|          25750|
|      Driven Growers|          19000|
+--------------------+---------------+

